# 检索流程诊断 Notebook

逐步执行检索 pipeline，查看每个阶段的结果：
1. 查看所有 chunk 和 embedding
2. Dense 向量检索（Milvus）
3. BM25 关键词检索
4. RRF 融合
5. Reranker 重排序
6. 阈值分析

修改 `TEST_QUERY` 变量来测试不同查询。

In [ ]:
import sys
import os
import json
import time
import numpy as np
import requests
from pathlib import Path
from collections import defaultdict

os.chdir(Path(os.getcwd()))  # ensure cwd is project root
sys.path.insert(0, str(Path.cwd()))

# ⚠️ CRITICAL: langchain_compat MUST be imported before any paddleocr import
from src.document_parser.langchain_compat import *

from config.settings import settings
from src.retrieval.vector_retriever import MilvusVectorRetriever
from src.retrieval.bm25_retriever import BM25Retriever, tokenize
from src.embeddings.bge_m3 import BGEM3Embedding
from src.storage import doc_store

# ── Cell 1: Config ──
TEST_QUERY = "每日工作时间"
FOLDER_FILTER = None   # e.g. "/" or "/mydocs" or None

print(f"✅ 模块加载完成")
print(f"   Test query: '{TEST_QUERY}'")
print(f"   Milvus: {settings.milvus_host}:{settings.milvus_port}")
print(f"   Collection: {settings.milvus_collection}")
print(f"   Chunk size: {settings.chunk_size}")
print(f"   Top K: {settings.hybrid_top_k}")
print(f"   RRF k: {settings.hybrid_rrf_k}")
print(f"   Dense weight: {settings.hybrid_dense_weight}")
print(f"   BM25 weight: {settings.hybrid_bm25_weight}")
print(f"   Threshold: {settings.retrieval_similarity_threshold}")
print(f"   Reranker: {settings.reranker_method}")

## Step 1: 查看所有 Chunk

从 Milvus 拉取所有 chunk，查看文档分布和文本内容。

In [ ]:
vr = MilvusVectorRetriever()
all_chunks = vr.fetch_all_chunks()

print(f"📊 总 chunk 数: {len(all_chunks)}")
print()

# 按文档分组
by_doc = defaultdict(list)
for c in all_chunks:
    by_doc[c.get('doc_id', '?')].append(c)

for doc_id, chunks in sorted(by_doc.items()):
    doc = doc_store.get_document(doc_id)
    fname = doc['filename'] if doc else doc_id
    print(f"📄 {fname} (doc_id={doc_id}): {len(chunks)} chunks")

# 搜索包含关键词的 chunk
keyword = TEST_QUERY
matching = []
for c in all_chunks:
    if keyword in c.get('text', ''):
        matching.append(c)
print(f"\n🔍 包含 '{keyword}' 的 chunk: {len(matching)} 个")
for m in matching[:10]:
    doc = doc_store.get_document(m['doc_id'])
    fname = doc['filename'] if doc else m['doc_id']
    print(f"  [{m['chunk_id']}] {fname}")
    print(f"    text[:200]: {m['text'][:200]}")
    print()

## Step 2: Embedding 查看

对查询生成 embedding，查看维度。也可以查看某个 chunk 的 embedding。

In [ ]:
emb = BGEM3Embedding()
query_vec = emb.encode_query_dense(TEST_QUERY)

print(f"🔢 Embedding dim: {query_vec.shape}")
print(f"   查询: '{TEST_QUERY}'")
print(f"   向量前10维: {query_vec[:10]}")
print(f"   向量范数: {np.linalg.norm(query_vec):.4f}")
print(f"   值域: [{query_vec.min():.4f}, {query_vec.max():.4f}]")

## Step 3: Dense 向量检索

用查询 embedding 在 Milvus 中搜索，查看余弦相似度得分。

In [ ]:
# 构建 doc filter
expr = None
if FOLDER_FILTER:
    ids = doc_store.get_doc_ids_by_filter(folder=FOLDER_FILTER)
    if ids:
        id_list = ", ".join(f'"{d}"' for d in ids)
        expr = f"doc_id in [{id_list}]"
        print(f"📂 Folder filter: {FOLDER_FILTER} → {len(ids)} docs\n")

dense_results = vr.search(query_vec, top_k=20, expr=expr)

print(f"🎯 Dense search (COSINE), top {len(dense_results)} results:\n")
for i, r in enumerate(dense_results):
    doc = doc_store.get_document(r['doc_id'])
    fname = doc['filename'] if doc else r['doc_id']
    score = r.get('score', 0)
    bar = '█' * int(score * 50) if score > 0 else ''
    print(f"  [{i+1:2d}] score={score:.4f} {bar}")
    print(f"       doc: {fname}")
    print(f"       chunk: {r['chunk_id']}")
    print(f"       text[:150]: {r['text'][:150]}")
    print()

## Step 4: BM25 关键词检索

使用 jieba 分词后进行 BM25 检索。

In [ ]:
bm25 = BM25Retriever()
bm25.load()

if bm25._model is None:
    print("⚠️ BM25 索引未找到，尝试从 Milvus 重建...")
    ok = bm25.rebuild_from_milvus()
    if not ok:
        print("❌ BM25 重建失败")

if bm25._model is not None:
    print(f"✅ BM25 已加载: {len(bm25._chunks)} chunks")
    print(f"   分词测试: {tokenize(TEST_QUERY)}")
    print()
    
    bm25_results = bm25.search(TEST_QUERY, top_k=20)
    
    print(f"🔑 BM25 search, top {len(bm25_results)} results:\n")
    for i, r in enumerate(bm25_results):
        doc = doc_store.get_document(r['doc_id'])
        fname = doc['filename'] if doc else r['doc_id']
        score = r.get('score', 0)
        bar = '█' * int(score * 50) if score > 0 else ''
        print(f"  [{i+1:2d}] score={score:.4f} {bar}")
        print(f"       doc: {fname}")
        print(f"       chunk: {r['chunk_id']}")
        print(f"       text[:150]: {r['text'][:150]}")
        print()
else:
    bm25_results = []
    print("❌ BM25 未加载")

# 检查关键词是否在 BM25 语料中
if bm25._chunks:
    kw_count = sum(1 for c in bm25._chunks if TEST_QUERY in c.get('text', ''))
    print(f"\n📊 关键词 '{TEST_QUERY}' 在 BM25 索引中出现: {kw_count} / {len(bm25._chunks)} 个 chunk")

## Step 5: RRF 融合

手动执行 RRF 算法，查看融合后的得分。绿色=通过阈值，红色=被过滤。

In [ ]:
k = settings.hybrid_rrf_k
dense_weight = settings.hybrid_dense_weight
bm25_weight = settings.hybrid_bm25_weight
threshold = settings.retrieval_similarity_threshold

scores_raw: dict[str, tuple[dict, float]] = {}
dense_scores: dict[str, float] = {}
bm25_scores: dict[str, float] = {}

for rank, doc in enumerate(dense_results):
    cid = doc.get('chunk_id', '')
    rrf = dense_weight / (k + rank + 1)
    scores_raw[cid] = (doc, scores_raw.get(cid, (doc, 0.0))[1] + rrf)
    dense_scores[cid] = doc.get('score', 0)

for rank, doc in enumerate(bm25_results):
    cid = doc.get('chunk_id', '')
    rrf = bm25_weight / (k + rank + 1)
    scores_raw[cid] = (doc, scores_raw.get(cid, (doc, 0.0))[1] + rrf)
    bm25_scores[cid] = doc.get('score', 0)

ranked_rrf = sorted(scores_raw.items(), key=lambda x: x[1][1], reverse=True)
max_rrf = ranked_rrf[0][1][1] if ranked_rrf else 1.0

print(f"⚖️ RRF Fusion (k={k}, dense_w={dense_weight}, bm25_w={bm25_weight})")
print(f"   Threshold: {threshold} (归一化后)")
print(f"   Dense 结果: {len(dense_results)}, BM25 结果: {len(bm25_results)}")
print(f"   融合后候选: {len(ranked_rrf)}")
print()

passed = 0
dropped = 0
for i, (cid, (doc, raw_score)) in enumerate(ranked_rrf):
    normalized = raw_score / max_rrf if max_rrf > 0 else 0
    passes = threshold == 0 or normalized >= threshold
    if passes:
        passed += 1
    else:
        dropped += 1
    
    doc_obj = doc_store.get_document(doc['doc_id'])
    fname = doc_obj['filename'] if doc_obj else doc['doc_id']
    
    icon = "🟢" if passes else "🔴"
    sources = []
    if cid in dense_scores:
        sources.append(f"D:{dense_scores[cid]:.4f}")
    if cid in bm25_scores:
        sources.append(f"B:{bm25_scores[cid]:.4f}")
    
    print(f"  {icon} [{i+1:2d}] RRF={normalized:.4f} (raw={raw_score:.6f}) {'✓' if passes else '✗ 被过滤'}")
    print(f"       {', '.join(sources)}")
    print(f"       {fname} :: {doc['chunk_id']}")
    print(f"       text[:120]: {doc['text'][:120]}")
    print()

print(f"📊 RRF 阶段: {passed} 通过 / {dropped} 被阈值 {threshold} 过滤")

## Step 6: Reranker 重排序

调用 SiliconFlow reranker API（或本地 fallback）。

In [ ]:
# 取 RRF 通过阈值的结果作为 reranker 输入
rrf_passed = [doc for cid, (doc, raw_score) in ranked_rrf
              if threshold == 0 or (max_rrf > 0 and (raw_score / max_rrf) >= threshold)]

if not rrf_passed:
    print("⚠️ RRF 阶段没有结果通过阈值，使用所有结果作为输入")
    rrf_passed = [doc for _, (doc, _) in ranked_rrf[:settings.hybrid_top_k]]

print(f"📥 Reranker 输入: {len(rrf_passed)} 条结果")

api_key = settings.siliconflow_api_key
if api_key:
    texts = [r.get('text', '') for r in rrf_passed]
    print(f"   调用 SiliconFlow reranker: {settings.siliconflow_rerank_model}")
    print(f"   URL: {settings.siliconflow_rerank_url}")
    print()
    
    t0 = time.time()
    try:
        resp = requests.post(
            settings.siliconflow_rerank_url,
            headers={
                "Authorization": f"Bearer {api_key}",
                "Content-Type": "application/json",
            },
            json={
                "model": settings.siliconflow_rerank_model,
                "query": TEST_QUERY,
                "documents": texts,
                "top_n": min(10, len(texts)),
                "return_documents": False,
            },
            timeout=30,
        )
        elapsed = (time.time() - t0) * 1000
        
        if resp.status_code == 200:
            data = resp.json()
            reranked_items = data.get('results', [])
            _rerank_scores = [item.get('relevance_score', 0) for item in reranked_items]
            print(f"✅ Reranker 响应 ({elapsed:.0f}ms), 返回 {len(reranked_items)} 条")
            print()
            
            for i, item in enumerate(reranked_items):
                idx = item.get('index', 0)
                score = item.get('relevance_score', 0)
                passes = threshold == 0 or score >= threshold
                
                if 0 <= idx < len(rrf_passed):
                    r = rrf_passed[idx]
                    doc_obj = doc_store.get_document(r['doc_id'])
                    fname = doc_obj['filename'] if doc_obj else r['doc_id']
                    
                    icon = "🟢" if passes else "🔴"
                    bar = '█' * int(score * 50)
                    print(f"  {icon} [{i+1:2d}] rerank_score={score:.4f} {bar} {'✓ 通过' if passes else '✗ 被过滤'}")
                    print(f"      原始索引: {idx}, {fname} :: {r['chunk_id']}")
                    print(f"      text[:120]: {r['text'][:120]}")
                    print()
        else:
            print(f"❌ Reranker API error: HTTP {resp.status_code}")
            print(f"   {resp.text[:500]}")
    except Exception as e:
        print(f"❌ Reranker 调用失败: {e}")
else:
    print("⚠️ SiliconFlow API key 为空，使用本地 cosine similarity 作为 reranker")
    from sklearn.metrics.pairwise import cosine_similarity
    chunk_vecs = emb.encode_dense([r['text'] for r in rrf_passed])
    sims = cosine_similarity(query_vec.reshape(1, -1), chunk_vecs).flatten()
    ranked = sorted(zip(rrf_passed, sims), key=lambda x: x[1], reverse=True)
    _rerank_scores = [sim for _, sim in ranked]
    for i, (r, sim) in enumerate(ranked[:10]):
        passes = threshold == 0 or sim >= threshold
        icon = "🟢" if passes else "🔴"
        print(f"  {icon} [{i+1:2d}] cos_sim={sim:.4f} {'✓' if passes else '✗'}")
        print(f"      {r['text'][:120]}")
        print()

## Step 7: 阈值分析 & 建议

分析当前阈值是否合理。

In [ ]:
print(f"{'='*60}")
print(f"  阈值诊断报告")
print(f"{'='*60}")
print(f"  当前阈值: {threshold}")
print()

# 分析 dense 分数分布
if dense_results:
    d_scores = [r['score'] for r in dense_results]
    print(f"  Dense (cosine) 分数分布:")
    print(f"    top1: {d_scores[0]:.4f}")
    print(f"    top5: {d_scores[min(4, len(d_scores)-1)]:.4f}")
    print(f"    top10: {d_scores[min(9, len(d_scores)-1)]:.4f}")
    print(f"    mean: {np.mean(d_scores):.4f}")
    print(f"    min: {np.min(d_scores):.4f}")
    print()

# 分析 BM25 分数分布
if bm25_results:
    b_scores = [r['score'] for r in bm25_results]
    print(f"  BM25 分数分布:")
    print(f"    top1: {b_scores[0]:.4f}")
    print(f"    top5: {b_scores[min(4, len(b_scores)-1)]:.4f}")
    print(f"    mean: {np.mean(b_scores):.4f}")
    print()

# 分析 RRF 归一化分数分布
if ranked_rrf:
    rrf_norm = [raw / max_rrf for _, (_, raw) in ranked_rrf]
    print(f"  RRF 归一化分数分布:")
    print(f"    top1:  {rrf_norm[0]:.4f}")
    print(f"    top3:  {rrf_norm[min(2, len(rrf_norm)-1)]:.4f}")
    print(f"    top5:  {rrf_norm[min(4, len(rrf_norm)-1)]:.4f}")
    print(f"    top10: {rrf_norm[min(9, len(rrf_norm)-1)]:.4f}")
    above_06 = sum(1 for s in rrf_norm if s >= 0.6)
    above_03 = sum(1 for s in rrf_norm if s >= 0.3)
    print(f"    ≥0.6: {above_06} 条")
    print(f"    ≥0.3: {above_03} 条")
    print()

# 分析 reranker 分数分布
rerank_scores_data = _rerank_scores if '_rerank_scores' in dir() else []
if rerank_scores_data:
    rs = rerank_scores_data
    print(f"  Reranker 分数分布:")
    print(f"    top1: {rs[0]:.4f}")
    print(f"    top5: {rs[min(4, len(rs)-1)]:.4f}")
    print(f"    mean: {np.mean(rs):.4f}")
    above_06 = sum(1 for s in rs if s >= 0.6)
    above_03 = sum(1 for s in rs if s >= 0.3)
    print(f"    ≥0.6: {above_06} 条")
    print(f"    ≥0.3: {above_03} 条")
    print()

# 建议
print("  💡 建议:")
if threshold > 0.3:
    print(f"     当前阈值 {threshold} 较高，建议降到 0.3 试试")
else:
    print(f"     阈值 {threshold} 合理")
print(f"     如果 RRF 阶段有很多结果但 reranker 阶段全被过滤，")
print(f"     说明 reranker 模型和你的数据特点不匹配")

## Step 8: 快速测试 — 自定义查询

修改 `TEST_QUERY` 变量（回到第一个 cell），重新运行所有 cell（Run All Cells）。